# IP Vertical Classification Churn Analysis

## Overview
This notebook analyzes changes in IP-to-vertical classifications between production and development environments. We use consistent sampling methodology to efficiently track how IP addresses are reclassified across different business verticals when transitioning from the current production model to a new development model.

## Key Definitions

### Core Concepts
- **IP (Internet Protocol Address)**: Unique identifier for devices on the internet. In this context, IPs represent user locations/devices that visit websites or use services.
- **Vertical**: A business category or industry segment (e.g., "Apparel & Accessories", "Gaming", "Current Affairs"). Each vertical represents a specific market segment.
- **Vertical ID**: A 6-digit numerical identifier for each vertical (e.g., "101000" for Apparel & Accessories).
- **IP-Vertical Association**: A mapping that classifies an IP address into one or more business verticals based on browsing behavior or other signals.

### Environments
- **Production (prod)**: The current live model actively classifying IPs
- **Development (dev)**: The new model being evaluated for deployment
- **Partition Date**: Both environments analyzed for dt='2025-07-07' to ensure fair comparison

### Churn Metrics
- **Retained**: IP-vertical pairs that exist in both prod and dev with the same classification
  - Example: IP 1.2.3.4 classified as "Gaming" in both environments
- **New**: IP-vertical pairs that exist only in dev (not in prod)
  - Example: IP 1.2.3.4 newly classified as "Current Affairs" in dev
- **Lost**: IP-vertical pairs that exist only in prod (not in dev)  
  - Example: IP 1.2.3.4 no longer classified as "Gambling" in dev
- **Changed**: IPs that exist in both but changed vertical classification

### Rate Calculations
- **Retention Rate** = (Retained IPs in vertical) / (Total Prod IPs in vertical) × 100
- **Churn Rate** = (Lost IPs from vertical) / (Total Prod IPs in vertical) × 100  
- **New Rate** = (New IPs to vertical) / (Total Dev IPs in vertical) × 100

### Sampling Methodology
- **Consistent Sampling**: We sample the same 1% of IPs from both environments using hash-based selection
- **Sample Size**: 2.17M IPs out of ~217M total unique IPs
- **Statistical Validity**: Results have 95% confidence interval with ±0.03% margin of error
- **Extrapolation**: Sample metrics are multiplied by 100 to estimate full population

## Business Context
IP-to-vertical classification is used to:
1. Understand user interests and behavior patterns
2. Enable targeted advertising and content recommendations
3. Provide market insights and audience analytics
4. Support business intelligence and strategic decisions

## Analysis Goals
1. Quantify the overall change between production and development models
2. Identify which verticals are most affected by the model change
3. Validate that changes align with business objectives
4. Ensure data quality and model stability

## Interpretation Guide
- **High Retention (>90%)**: Vertical classification is stable and consistent
- **High Churn (>50%)**: Significant reclassification occurring, requires validation
- **Large Net Changes**: May indicate model refocus or new classification criteria
- **Multiple Verticals per IP**: Normal - IPs can have diverse interests

## Performance Notes
- Full dataset: ~200M+ unique IPs, ~1B+ IP-vertical associations
- 1% sampling reduces runtime from 6+ hours to ~70 minutes
- Hash-based sampling ensures reproducibility and no bias

In [0]:
from pyspark.sql import functions as F
import time
from datetime import datetime

print("="*80)
print("CONSISTENT IP SAMPLING FOR CHURN ANALYSIS")
print("="*80)

# Sampling configuration
SAMPLE_RATE = 1.0  # 1% sample - adjust as needed (0.01 to 0.10)
RANDOM_SEED = 42   # For reproducibility

print(f"Sample Rate: {SAMPLE_RATE * 100}%")
print(f"Random Seed: {RANDOM_SEED}")
print(f"Analysis Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Spark optimizations
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "500")

start_time = time.time()


CONSISTENT IP SAMPLING FOR CHURN ANALYSIS
Sample Rate: 100.0%
Random Seed: 42
Analysis Start: 2025-07-22 22:22:05


In [0]:
# Cell 0: Test S3 Write Access
print("Testing S3 write permissions...")
print("="*80)

OUTPUT_PATH = "s3://mntn-data-archive-dev/malachi/ip-33"
TEST_PATH = f"{OUTPUT_PATH}/test_write"

try:
    # Create a simple test DataFrame
    test_df = spark.createDataFrame(
        [(1, "test", "2025-07-22")], 
        ["id", "message", "date"]
    )
    
    # Try to write to S3 as parquet
    test_df.write.mode("overwrite").parquet(TEST_PATH)
    
    print(f"✓ Successfully wrote test file to: {TEST_PATH}")
    print("✓ S3 write permissions confirmed")
    
    # Clean up test file
    try:
        dbutils.fs.rm(TEST_PATH, recurse=True)
        print("✓ Test file cleaned up")
    except:
        print("! Could not clean up test file (not critical)")
        
except Exception as e:
    print(f"✗ ERROR: Cannot write to S3 bucket")
    print(f"✗ Error details: {str(e)}")
    print("\nPlease check:")
    print("1. S3 bucket exists and path is correct")
    print("2. You have write permissions to the bucket")
    print("3. AWS credentials are properly configured")
    raise e

print("\n" + "="*80)
print("Proceeding with analysis...")

Testing S3 write permissions...
✓ Successfully wrote test file to: s3://mntn-data-archive-dev/malachi/ip-33/test_write
✓ S3 write permissions confirmed
✓ Test file cleaned up

Proceeding with analysis...


In [0]:
print("\nSTEP 1: Creating Universe of All Unique IPs (Optimized)")
print("="*50)

# Define output base path
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Use approximate distinct count first to get the scale
approx_prod_ips = spark.sql("""
    SELECT approx_count_distinct(ip) as approx_ips
    FROM prod.ml.ip_vertical_associations
    WHERE dt = '2025-07-21'
""").collect()[0]['approx_ips']

approx_dev_ips = spark.sql("""
    SELECT approx_count_distinct(ip) as approx_ips
    FROM dev.ml.ip_vertical_associations
    WHERE dt = '2025-07-21'
""").collect()[0]['approx_ips']

print(f"Approximate unique IPs - Prod: {approx_prod_ips:,}, Dev: {approx_dev_ips:,}")

# Instead of creating full universe, sample directly from source tables
print("\nCreating consistent sample using hash-based selection...")

# Define hash-based sampling function
sample_condition = f"ABS(HASH(ip)) % 10000 < {int(SAMPLE_RATE * 10000)}"

# Count sampled IPs for statistics
sampled_prod_ips = spark.sql(f"""
    SELECT COUNT(DISTINCT ip) as count
    FROM prod.ml.ip_vertical_associations
    WHERE dt = '2025-07-21' AND {sample_condition}
""").collect()[0]['count']

sampled_dev_ips = spark.sql(f"""
    SELECT COUNT(DISTINCT ip) as count
    FROM dev.ml.ip_vertical_associations  
    WHERE dt = '2025-07-21' AND {sample_condition}
""").collect()[0]['count']

sampled_union = spark.sql(f"""
    SELECT COUNT(DISTINCT ip) as count
    FROM (
        SELECT DISTINCT ip 
        FROM prod.ml.ip_vertical_associations
        WHERE dt = '2025-07-21' AND {sample_condition}
        
        UNION
        
        SELECT DISTINCT ip
        FROM dev.ml.ip_vertical_associations
        WHERE dt = '2025-07-21' AND {sample_condition}
    )
""").collect()[0]['count']

print(f"\nSampled IPs - Prod: {sampled_prod_ips:,}, Dev: {sampled_dev_ips:,}, Union: {sampled_union:,}")
print(f"Estimated total unique IPs: ~{int(sampled_union / SAMPLE_RATE):,}")

# Store the sampling condition for reuse
spark.conf.set("sampling.condition", sample_condition)

# Save sampling statistics
sampling_stats_df = spark.createDataFrame([
    {
        "metric": "approx_prod_ips",
        "value": approx_prod_ips,
        "sample_rate": SAMPLE_RATE,
        "run_timestamp": RUN_DATE
    },
    {
        "metric": "approx_dev_ips", 
        "value": approx_dev_ips,
        "sample_rate": SAMPLE_RATE,
        "run_timestamp": RUN_DATE
    },
    {
        "metric": "sampled_prod_ips",
        "value": sampled_prod_ips,
        "sample_rate": SAMPLE_RATE,
        "run_timestamp": RUN_DATE
    },
    {
        "metric": "sampled_dev_ips",
        "value": sampled_dev_ips,
        "sample_rate": SAMPLE_RATE,
        "run_timestamp": RUN_DATE
    },
    {
        "metric": "sampled_union_ips",
        "value": sampled_union,
        "sample_rate": SAMPLE_RATE,
        "run_timestamp": RUN_DATE
    }
])

sampling_stats_path = f"{OUTPUT_BASE}/sampling_stats/{RUN_DATE}"
sampling_stats_df.write.mode("overwrite").parquet(sampling_stats_path)
print(f"\n✓ Saved sampling statistics to: {sampling_stats_path}")


STEP 1: Creating Universe of All Unique IPs (Optimized)
Approximate unique IPs - Prod: 185,182,062, Dev: 204,211,860

Creating consistent sample using hash-based selection...

Sampled IPs - Prod: 192,945,321, Dev: 211,201,487, Union: 219,613,393
Estimated total unique IPs: ~219,613,393

✓ Saved sampling statistics to: s3://mntn-data-archive-dev/malachi/ip-33/sampling_stats/20250722_222229


In [0]:
print("\nSTEPS 3 & 4: Extracting Sampled Data (Combined)")
print("="*50)

# Load vertical names first
from pyspark.sql.functions import broadcast
verticals_df = spark.read.parquet("s3://mntn-data-archive-prod/vertical_categorizations/advertiser_verticals/")
verticals_df = broadcast(verticals_df)  # Broadcast for better performance
verticals_df.createOrReplaceTempView("vertical_names")

# Get the sampling condition from Cell 2
sample_condition = spark.conf.get("sampling.condition")
print(f"Using sampling condition: {sample_condition}")

# Get run date from previous cell
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Production data
prod_sampled = spark.sql(f"""
    SELECT DISTINCT 
        ip,
        CAST(FLOOR(data_source_category_id) AS STRING) as vertical_id
    FROM prod.ml.ip_vertical_associations
    WHERE dt = '2025-07-21'
        AND LENGTH(CAST(FLOOR(data_source_category_id) AS STRING)) = 6
        AND {sample_condition}
""")
prod_sampled.cache()
prod_sampled.createOrReplaceTempView("prod_sampled")

# Save prod_sampled to parquet
prod_sampled_path = f"{OUTPUT_BASE}/prod_sampled/{RUN_DATE}"
prod_sampled.write.mode("overwrite").parquet(prod_sampled_path)
print(f"✓ Saved prod_sampled to: {prod_sampled_path}")

# Development data  
dev_sampled = spark.sql(f"""
    SELECT DISTINCT 
        ip,
        CAST(FLOOR(data_source_category_id) AS STRING) as vertical_id
    FROM dev.ml.ip_vertical_associations
    WHERE dt = '2025-07-21'
        AND LENGTH(CAST(FLOOR(data_source_category_id) AS STRING)) = 6
        AND {sample_condition}
""")
dev_sampled.cache()
dev_sampled.createOrReplaceTempView("dev_sampled")

# Save dev_sampled to parquet
dev_sampled_path = f"{OUTPUT_BASE}/dev_sampled/{RUN_DATE}"
dev_sampled.write.mode("overwrite").parquet(dev_sampled_path)
print(f"✓ Saved dev_sampled to: {dev_sampled_path}")

# Create sampled_ips view for later cells
sampled_ips = spark.sql(f"""
    SELECT DISTINCT ip FROM (
        SELECT ip FROM prod_sampled
        UNION
        SELECT ip FROM dev_sampled
    )
""")
sampled_ips.createOrReplaceTempView("sampled_ips")

# Save sampled_ips to parquet
sampled_ips_path = f"{OUTPUT_BASE}/sampled_ips/{RUN_DATE}"
sampled_ips.write.mode("overwrite").parquet(sampled_ips_path)
print(f"✓ Saved sampled_ips to: {sampled_ips_path}")

# Get statistics for both in one query
stats = spark.sql("""
    WITH prod_stats AS (
        SELECT 
            'prod' as env,
            COUNT(DISTINCT ip) as unique_ips,
            COUNT(DISTINCT vertical_id) as unique_verticals,
            COUNT(*) as ip_vertical_pairs
        FROM prod_sampled
    ),
    dev_stats AS (
        SELECT 
            'dev' as env,
            COUNT(DISTINCT ip) as unique_ips,
            COUNT(DISTINCT vertical_id) as unique_verticals,
            COUNT(*) as ip_vertical_pairs
        FROM dev_sampled
    )
    SELECT * FROM prod_stats
    UNION ALL
    SELECT * FROM dev_stats
""").collect()

# Save environment stats
env_stats_df = spark.createDataFrame(stats)
env_stats_path = f"{OUTPUT_BASE}/environment_stats/{RUN_DATE}"
env_stats_df.write.mode("overwrite").parquet(env_stats_path)
print(f"✓ Saved environment stats to: {env_stats_path}")

for row in stats:
    print(f"\n{row['env'].upper()} Sample Statistics:")
    print(f"- Unique IPs: {row['unique_ips']:,}")
    print(f"- Unique verticals: {row['unique_verticals']:,}")
    print(f"- IP-vertical pairs: {row['ip_vertical_pairs']:,}")


STEPS 3 & 4: Extracting Sampled Data (Combined)
Using sampling condition: ABS(HASH(ip)) % 10000 < 10000
✓ Saved prod_sampled to: s3://mntn-data-archive-dev/malachi/ip-33/prod_sampled/20250722_224230
✓ Saved dev_sampled to: s3://mntn-data-archive-dev/malachi/ip-33/dev_sampled/20250722_224230
✓ Saved sampled_ips to: s3://mntn-data-archive-dev/malachi/ip-33/sampled_ips/20250722_224230
✓ Saved environment stats to: s3://mntn-data-archive-dev/malachi/ip-33/environment_stats/20250722_224230

PROD Sample Statistics:
- Unique IPs: 192,945,321
- Unique verticals: 148
- IP-vertical pairs: 1,002,497,236

DEV Sample Statistics:
- Unique IPs: 211,201,487
- Unique verticals: 148
- IP-vertical pairs: 1,128,521,950


In [0]:
print("\nSTEP 5: Calculating Churn Statistics on Sample")
print("="*50)

# Get paths from previous cells
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Perform churn analysis on the sampled data
churn_analysis = spark.sql("""
    WITH churn_detail AS (
        SELECT 
            COALESCE(p.ip, d.ip) as ip,
            COALESCE(p.vertical_id, d.vertical_id) as vertical_id,
            p.ip as prod_ip,
            d.ip as dev_ip,
            CASE 
                WHEN p.ip IS NOT NULL AND d.ip IS NOT NULL THEN 'retained'
                WHEN p.ip IS NULL THEN 'new'
                WHEN d.ip IS NULL THEN 'lost'
            END as status
        FROM prod_sampled p
        FULL OUTER JOIN dev_sampled d
            ON p.ip = d.ip AND p.vertical_id = d.vertical_id
    ),
    vertical_summary AS (
        SELECT 
            vertical_id,
            COUNT(CASE WHEN status = 'retained' THEN 1 END) as retained,
            COUNT(CASE WHEN status = 'new' THEN 1 END) as new,
            COUNT(CASE WHEN status = 'lost' THEN 1 END) as lost,
            COUNT(CASE WHEN prod_ip IS NOT NULL THEN 1 END) as prod_total,
            COUNT(CASE WHEN dev_ip IS NOT NULL THEN 1 END) as dev_total
        FROM churn_detail
        GROUP BY vertical_id
    )
    SELECT 
        v.*,
        vn.vertical_name,
        ROUND(100.0 * v.retained / NULLIF(v.prod_total, 0), 2) as retention_rate,
        ROUND(100.0 * v.lost / NULLIF(v.prod_total, 0), 2) as churn_rate,
        ROUND(100.0 * v.new / NULLIF(v.dev_total, 0), 2) as new_rate
    FROM vertical_summary v
    LEFT JOIN vertical_names vn ON v.vertical_id = CAST(vn.vertical_id AS STRING)
    ORDER BY v.new DESC
""")

churn_analysis.cache()
churn_analysis.createOrReplaceTempView("churn_results")

# Save churn results to parquet
churn_results_path = f"{OUTPUT_BASE}/churn_results/{RUN_DATE}"
churn_analysis.write.mode("overwrite").parquet(churn_results_path)
print(f"✓ Saved churn results to: {churn_results_path}")

# Also save the detailed churn data (before aggregation)
churn_detail_df = spark.sql("""
    SELECT 
        COALESCE(p.ip, d.ip) as ip,
        COALESCE(p.vertical_id, d.vertical_id) as vertical_id,
        p.ip as prod_ip,
        d.ip as dev_ip,
        CASE 
            WHEN p.ip IS NOT NULL AND d.ip IS NOT NULL THEN 'retained'
            WHEN p.ip IS NULL THEN 'new'
            WHEN d.ip IS NULL THEN 'lost'
        END as status
    FROM prod_sampled p
    FULL OUTER JOIN dev_sampled d
        ON p.ip = d.ip AND p.vertical_id = d.vertical_id
""")

churn_detail_path = f"{OUTPUT_BASE}/churn_detail/{RUN_DATE}"
churn_detail_df.write.mode("overwrite").parquet(churn_detail_path)
print(f"✓ Saved churn detail to: {churn_detail_path}")

# Overall statistics
overall_stats = spark.sql("""
    SELECT 
        SUM(retained) as total_retained,
        SUM(new) as total_new,
        SUM(lost) as total_lost,
        SUM(prod_total) as total_prod,
        SUM(dev_total) as total_dev
    FROM churn_results
""").collect()[0]

# Save overall stats
overall_stats_df = spark.createDataFrame([overall_stats])
overall_stats_path = f"{OUTPUT_BASE}/overall_stats/{RUN_DATE}"
overall_stats_df.write.mode("overwrite").parquet(overall_stats_path)
print(f"✓ Saved overall stats to: {overall_stats_path}")

print("\nOverall Churn Statistics (Sample):")
print("-" * 40)
print(f"Total prod IP-vertical pairs: {overall_stats['total_prod']:,}")
print(f"Total dev IP-vertical pairs: {overall_stats['total_dev']:,}")
print(f"Retained: {overall_stats['total_retained']:,} ({overall_stats['total_retained']/overall_stats['total_prod']*100:.1f}%)")
print(f"New: {overall_stats['total_new']:,} ({overall_stats['total_new']/overall_stats['total_dev']*100:.1f}%)")
print(f"Lost: {overall_stats['total_lost']:,} ({overall_stats['total_lost']/overall_stats['total_prod']*100:.1f}%)")

print("\nEstimated Full Population Statistics:")
print("-" * 40)
scale_factor = 1 / SAMPLE_RATE
print(f"Est. total prod IP-vertical pairs: {int(overall_stats['total_prod'] * scale_factor):,}")
print(f"Est. total dev IP-vertical pairs: {int(overall_stats['total_dev'] * scale_factor):,}")
print(f"Est. retained: {int(overall_stats['total_retained'] * scale_factor):,}")
print(f"Est. new: {int(overall_stats['total_new'] * scale_factor):,}")
print(f"Est. lost: {int(overall_stats['total_lost'] * scale_factor):,}")


STEP 5: Calculating Churn Statistics on Sample
✓ Saved churn results to: s3://mntn-data-archive-dev/malachi/ip-33/churn_results/20250722_230908
✓ Saved churn detail to: s3://mntn-data-archive-dev/malachi/ip-33/churn_detail/20250722_230908
✓ Saved overall stats to: s3://mntn-data-archive-dev/malachi/ip-33/overall_stats/20250722_230908

Overall Churn Statistics (Sample):
----------------------------------------
Total prod IP-vertical pairs: 1,006,615,173
Total dev IP-vertical pairs: 1,131,010,827
Retained: 730,481,583 (72.6%)
New: 400,529,244 (35.4%)
Lost: 276,133,590 (27.4%)

Estimated Full Population Statistics:
----------------------------------------
Est. total prod IP-vertical pairs: 1,006,615,173
Est. total dev IP-vertical pairs: 1,131,010,827
Est. retained: 730,481,583
Est. new: 400,529,244
Est. lost: 276,133,590


In [0]:
print("\nSTEP 6: Detailed Vertical Analysis")
print("="*50)

# Get paths from previous cells
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Top gainers and losers
print("Verticals by NEW IPs (Sample):")
top_gainers_df = spark.sql("""
    SELECT 
        vertical_name,
        prod_total,
        dev_total,
        new,
        lost,
        retained,
        retention_rate,
        churn_rate,
        dev_total - prod_total as net_change
    FROM churn_results
    WHERE prod_total > 100  -- Filter for meaningful sample sizes
    ORDER BY new DESC
""")
top_gainers_df.show(truncate=False)

# Save top gainers
top_gainers_path = f"{OUTPUT_BASE}/top_gainers/{RUN_DATE}"
top_gainers_df.write.mode("overwrite").parquet(top_gainers_path)
print(f"✓ Saved top gainers to: {top_gainers_path}")

print("\nVerticals by CHURN RATE (Sample):")
top_churners_df = spark.sql("""
    SELECT 
        vertical_name,
        prod_total,
        dev_total,
        retained,
        retention_rate,
        churn_rate
    FROM churn_results
    WHERE prod_total > 100  -- Filter for meaningful sample sizes
    ORDER BY churn_rate DESC
""")
top_churners_df.show(truncate=False)

# Save top churners
top_churners_path = f"{OUTPUT_BASE}/top_churners/{RUN_DATE}"
top_churners_df.write.mode("overwrite").parquet(top_churners_path)
print(f"✓ Saved top churners to: {top_churners_path}")

# Check for anomalies
high_churn_verticals = spark.sql("""
    SELECT COUNT(*) as count
    FROM churn_results
    WHERE churn_rate > 95 AND prod_total > 100
""").collect()[0]['count']

print(f"\nVerticals with >95% churn (min 100 sampled IPs): {high_churn_verticals}")

# IP movement patterns
print("\nIP Movement Analysis (Sample):")
ip_movements_df = spark.sql("""
    WITH ip_changes AS (
        SELECT 
            s.ip,
            COUNT(DISTINCT p.vertical_id) as prod_verticals,
            COUNT(DISTINCT d.vertical_id) as dev_verticals
        FROM sampled_ips s
        LEFT JOIN prod_sampled p ON s.ip = p.ip
        LEFT JOIN dev_sampled d ON s.ip = d.ip
        GROUP BY s.ip
    )
    SELECT 
        CASE 
            WHEN prod_verticals = 0 THEN 'New IPs (not in prod)'
            WHEN dev_verticals = 0 THEN 'Lost IPs (not in dev)'
            WHEN prod_verticals = 1 AND dev_verticals = 1 THEN 'Single vertical (both)'
            ELSE 'Multiple verticals'
        END as ip_category,
        COUNT(*) as ip_count
    FROM ip_changes
    GROUP BY 
        CASE 
            WHEN prod_verticals = 0 THEN 'New IPs (not in prod)'
            WHEN dev_verticals = 0 THEN 'Lost IPs (not in dev)'
            WHEN prod_verticals = 1 AND dev_verticals = 1 THEN 'Single vertical (both)'
            ELSE 'Multiple verticals'
        END
    ORDER BY ip_count DESC
""")
ip_movements_df.show()

# Save IP movements
ip_movements_path = f"{OUTPUT_BASE}/ip_movements/{RUN_DATE}"
ip_movements_df.write.mode("overwrite").parquet(ip_movements_path)
print(f"✓ Saved IP movements to: {ip_movements_path}")

# Also save the detailed IP changes data
ip_changes_detail_df = spark.sql("""
    SELECT 
        s.ip,
        COUNT(DISTINCT p.vertical_id) as prod_verticals,
        COUNT(DISTINCT d.vertical_id) as dev_verticals,
        COLLECT_SET(p.vertical_id) as prod_vertical_list,
        COLLECT_SET(d.vertical_id) as dev_vertical_list
    FROM sampled_ips s
    LEFT JOIN prod_sampled p ON s.ip = p.ip
    LEFT JOIN dev_sampled d ON s.ip = d.ip
    GROUP BY s.ip
""")

ip_changes_detail_path = f"{OUTPUT_BASE}/ip_changes_detail/{RUN_DATE}"
ip_changes_detail_df.write.mode("overwrite").parquet(ip_changes_detail_path)
print(f"✓ Saved IP changes detail to: {ip_changes_detail_path}")


STEP 6: Detailed Vertical Analysis
Top 15 Verticals by NEW IPs (Sample):
+-------------------------------------------+----------+---------+--------+-------+--------+--------------+----------+----------+
|vertical_name                              |prod_total|dev_total|new     |lost   |retained|retention_rate|churn_rate|net_change|
+-------------------------------------------+----------+---------+--------+-------+--------+--------------+----------+----------+
|Current Affairs                            |9041979   |66057556 |57644086|628509 |8413470 |93.05         |6.95      |57015577  |
|Games & Comics                             |1336402   |35782071 |34468753|23084  |1313318 |98.27         |1.73      |34445669  |
|Sewing, Arts, & Crafts                     |9629356   |30501643 |20953655|81368  |9547988 |99.16         |0.84      |20872287  |
|Music & Instruments                        |8695389   |24164845 |15729088|259632 |8435757 |97.01         |2.99      |15469456  |
|Learning & Educ

In [0]:
print("\nSTEP 7: Statistical Confidence and Final Report")
print("="*50)

# Get paths from previous cells
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Calculate confidence intervals for key metrics
import math

# Get sampled_ip_count from Cell 2 results
sampled_ip_count = sampled_union  # This variable is from Cell 2

def calculate_confidence_interval(p, n, confidence=0.95):
    """Calculate confidence interval for a proportion"""
    z = 1.96  # 95% confidence
    margin = z * math.sqrt((p * (1 - p)) / n)
    return (p - margin, p + margin)

# Overall retention rate confidence interval
retention_rate = overall_stats['total_retained'] / overall_stats['total_prod']
n_sample = overall_stats['total_prod']
ci_lower, ci_upper = calculate_confidence_interval(retention_rate, n_sample)

print("Statistical Confidence (95% CI):")
print("-" * 40)
print(f"Sample retention rate: {retention_rate*100:.2f}%")
print(f"95% Confidence Interval: [{ci_lower*100:.2f}%, {ci_upper*100:.2f}%]")
print(f"Margin of error: ±{(ci_upper - retention_rate)*100:.2f}%")

# Execution time
execution_time = (time.time() - start_time) / 60
print(f"\nExecution Time: {execution_time:.2f} minutes")
print(f"Speed improvement: ~{int(1/SAMPLE_RATE)}x faster than full analysis")

# Create summary statistics dataframe
summary_stats_df = spark.createDataFrame([{
    "run_timestamp": RUN_DATE,
    "sample_rate": SAMPLE_RATE,
    "sampled_ip_count": sampled_ip_count,
    "execution_time_minutes": execution_time,
    "overall_retention_rate": retention_rate,
    "overall_churn_rate": overall_stats['total_lost']/overall_stats['total_prod'],
    "new_ip_rate": overall_stats['total_new']/overall_stats['total_dev'],
    "confidence_interval_lower": ci_lower,
    "confidence_interval_upper": ci_upper,
    "margin_of_error": ci_upper - retention_rate,
    "high_churn_verticals_count": high_churn_verticals,
    "total_prod_pairs": overall_stats['total_prod'],
    "total_dev_pairs": overall_stats['total_dev'],
    "est_full_prod_pairs": int(overall_stats['total_prod'] * scale_factor),
    "est_full_dev_pairs": int(overall_stats['total_dev'] * scale_factor)
}])

summary_stats_path = f"{OUTPUT_BASE}/summary_stats/{RUN_DATE}"
summary_stats_df.write.mode("overwrite").parquet(summary_stats_path)
print(f"\n✓ Saved summary statistics to: {summary_stats_path}")

# Summary for JIRA
print("\n" + "="*80)
print("SUMMARY FOR JIRA REPORT")
print("="*80)

print(f"""
CHURN ANALYSIS SUMMARY - {datetime.now().strftime('%Y-%m-%d')}

Methodology: Consistent IP Sampling
- Sample Rate: {SAMPLE_RATE * 100}%
- Sample Size: {sampled_ip_count:,} IPs
- Execution Time: {execution_time:.1f} minutes

OVERALL METRICS:
- Overall Retention Rate: {retention_rate*100:.1f}% (±{(ci_upper - retention_rate)*100:.1f}%)
- Overall Churn Rate: {(overall_stats['total_lost']/overall_stats['total_prod'])*100:.1f}%
- New IP Rate: {(overall_stats['total_new']/overall_stats['total_dev'])*100:.1f}%

ESTIMATED FULL POPULATION:
- Total Prod IP-Verticals: ~{int(overall_stats['total_prod'] * scale_factor):,}
- Total Dev IP-Verticals: ~{int(overall_stats['total_dev'] * scale_factor):,}
- Net Change: ~{int((overall_stats['total_dev'] - overall_stats['total_prod']) * scale_factor):,}

HIGH-LEVEL FINDINGS:
- {high_churn_verticals} verticals show >95% churn rate
- Average retention across verticals: {retention_rate*100:.1f}%
""")

# Export top movers
print("\nTOP MOVERS (CSV Format):")
print("vertical_name,sample_prod,sample_dev,retention_rate,churn_rate,est_full_prod,est_full_dev")

top_movers = spark.sql("""
    SELECT 
        vertical_name,
        prod_total,
        dev_total,
        retention_rate,
        churn_rate
    FROM churn_results
    WHERE prod_total > 100
    ORDER BY ABS(dev_total - prod_total) DESC
    LIMIT 10
""").collect()

# Create top movers dataframe for saving
top_movers_data = []
for row in top_movers:
    est_prod = int(row['prod_total'] * scale_factor)
    est_dev = int(row['dev_total'] * scale_factor)
    print(f"{row['vertical_name']},{row['prod_total']},{row['dev_total']},{row['retention_rate']},{row['churn_rate']},{est_prod},{est_dev}")
    
    top_movers_data.append({
        "vertical_name": row['vertical_name'],
        "sample_prod": row['prod_total'],
        "sample_dev": row['dev_total'],
        "retention_rate": row['retention_rate'],
        "churn_rate": row['churn_rate'],
        "est_full_prod": est_prod,
        "est_full_dev": est_dev
    })

top_movers_df = spark.createDataFrame(top_movers_data)
top_movers_path = f"{OUTPUT_BASE}/top_movers_summary/{RUN_DATE}"
top_movers_df.write.mode("overwrite").parquet(top_movers_path)
print(f"\n✓ Saved top movers summary to: {top_movers_path}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

# Print summary of all saved files
print(f"\n📁 All outputs saved to: {OUTPUT_BASE}")
print(f"📅 Run timestamp: {RUN_DATE}")
print("\nSaved datasets:")
print(f"  - sampling_stats/{RUN_DATE}")
print(f"  - prod_sampled/{RUN_DATE}")
print(f"  - dev_sampled/{RUN_DATE}")
print(f"  - sampled_ips/{RUN_DATE}")
print(f"  - environment_stats/{RUN_DATE}")
print(f"  - churn_results/{RUN_DATE}")
print(f"  - churn_detail/{RUN_DATE}")
print(f"  - overall_stats/{RUN_DATE}")
print(f"  - top_gainers/{RUN_DATE}")
print(f"  - top_churners/{RUN_DATE}")
print(f"  - ip_movements/{RUN_DATE}")
print(f"  - ip_changes_detail/{RUN_DATE}")
print(f"  - summary_stats/{RUN_DATE}")
print(f"  - top_movers_summary/{RUN_DATE}")

# Cleanup
spark.catalog.clearCache()


STEP 7: Statistical Confidence and Final Report
Statistical Confidence (95% CI):
----------------------------------------
Sample retention rate: 72.57%
95% Confidence Interval: [72.57%, 72.57%]
Margin of error: ±0.00%

Execution Time: 83.93 minutes
Speed improvement: ~1x faster than full analysis

SUMMARY FOR JIRA REPORT

CHURN ANALYSIS SUMMARY - 2025-07-22

Methodology: Consistent IP Sampling
- Sample Rate: 100.0%
- Sample Size: 219,613,393 IPs
- Execution Time: 83.9 minutes

OVERALL METRICS:
- Overall Retention Rate: 72.6% (±0.0%)
- Overall Churn Rate: 27.4%
- New IP Rate: 35.4%

ESTIMATED FULL POPULATION:
- Total Prod IP-Verticals: ~1,006,615,173
- Total Dev IP-Verticals: ~1,131,010,827
- Net Change: ~124,395,654

HIGH-LEVEL FINDINGS:
- 0 verticals show >95% churn rate
- Average retention across verticals: 72.6%


TOP MOVERS (CSV Format):
vertical_name,sample_prod,sample_dev,retention_rate,churn_rate,est_full_prod,est_full_dev
Current Affairs,9041979,66057556,93.05,6.95,9041979,660

In [0]:
print("\nSTEP 8: Exporting All Vertical Summaries to S3")
print("="*50)

# Get paths from previous cells
OUTPUT_BASE = "s3://mntn-data-archive-dev/malachi/ip-33"
RUN_DATE = datetime.now().strftime('%Y%m%d_%H%M%S')

# Get the churn results with all metrics
vertical_summary_df = spark.sql("""
    SELECT 
        vertical_id,
        vertical_name,
        prod_total,
        dev_total,
        retained,
        new,
        lost,
        retention_rate,
        churn_rate,
        new_rate,
        dev_total - prod_total as net_change
    FROM churn_results
    ORDER BY new DESC
""")

# Save complete vertical summary
vertical_summary_path = f"{OUTPUT_BASE}/vertical_summary_complete/{RUN_DATE}"
vertical_summary_df.write.mode("overwrite").parquet(vertical_summary_path)
print(f"✓ Successfully exported complete vertical summaries to:")
print(f"  {vertical_summary_path}")

# Also create a more readable version with just key metrics
key_metrics_df = spark.sql("""
    SELECT 
        vertical_name,
        prod_total as prod_count,
        dev_total as dev_count,
        retention_rate,
        churn_rate,
        new_rate,
        dev_total - prod_total as net_change
    FROM churn_results
    WHERE prod_total > 100  -- Filter for meaningful sample sizes
    ORDER BY ABS(dev_total - prod_total) DESC
""")

key_metrics_path = f"{OUTPUT_BASE}/vertical_key_metrics/{RUN_DATE}"
key_metrics_df.write.mode("overwrite").parquet(key_metrics_path)
print(f"✓ Also exported key metrics summary to:")
print(f"  {key_metrics_path}")

# Show preview of what was saved
print("\nPreview of exported data (top 10 verticals by net change):")
key_metrics_df.limit(10).show(truncate=False)

print(f"\n🎉 All data successfully saved as parquet files!")
print(f"📁 Base path: {OUTPUT_BASE}")
print(f"📅 Run ID: {RUN_DATE}")

In [0]:
# Method 1: Count the rows
row_count = vertical_summary_df.count()
print(f"Total rows in vertical_summary_df: {row_count}")

java.util.concurrent.TimeoutException: Futures timed out after [30 seconds]
	at scala.concurrent.impl.Promise$DefaultPromise.ready(Promise.scala:259)
	at scala.concurrent.impl.Promise$DefaultPromise.result(Promise.scala:263)
	at scala.concurrent.Await$.$anonfun$result$1(package.scala:223)
	at scala.concurrent.BlockContext$DefaultBlockContext$.blockOn(BlockContext.scala:57)
	at scala.concurrent.Await$.result(package.scala:146)
	at com.databricks.backend.daemon.driver.JupyterKernelListener$RequestStatus.waitForReply(JupyterKernelListener.scala:315)
	at com.databricks.backend.daemon.driver.JupyterKernelListener.executeCommand(JupyterKernelListener.scala:1538)
	at com.databricks.backend.daemon.driver.JupyterDriverLocal.executePython(JupyterDriverLocal.scala:1640)
	at com.databricks.backend.daemon.driver.JupyterDriverLocal.repl(JupyterDriverLocal.scala:1502)
	at com.databricks.backend.daemon.driver.DriverLocal.$anonfun$execute$36(DriverLocal.scala:1310)
	at com.databricks.unity.UCSEphemeral